In [3]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Hugging Face token:", "OK" if HF_TOKEN else "Missing")

Hugging Face token: OK


In [4]:
%pip install -q pyannote.audio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [5]:
import torch
import numpy
import pyannote.audio

from importlib.metadata import version

print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("pyannote.audio:", version("pyannote.audio"))

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
NumPy: 2.4.6
pyannote.audio: 4.0.7
CUDA available: True
GPU: Tesla T4


In [6]:
import torch
from pyannote.audio import Pipeline
from google.colab import userdata

# Retrieve the token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("Hugging Face token is missing.")

MODEL_ID = "pyannote/speaker-diarization-community-1"

print("Loading speaker diarization model...")

pipeline = Pipeline.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

if pipeline is None:
    raise RuntimeError("Failed to load the diarization pipeline.")

# Move the model to GPU
pipeline.to(torch.device("cuda"))

print("Model loaded successfully.")
print("Inference device: GPU")

Loading speaker diarization model...


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

Model loaded successfully.
Inference device: GPU


In [7]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No audio file uploaded.")

audio_path = next(iter(uploaded))

print("Uploaded audio:", audio_path)

Saving meeting_test.wav to meeting_test.wav
Uploaded audio: meeting_test.wav


In [8]:
import soundfile as sf

audio_info = sf.info(audio_path)

print("Audio file:", audio_path)
print("Duration:", round(audio_info.duration, 2), "seconds")
print("Sample rate:", audio_info.samplerate, "Hz")
print("Channels:", audio_info.channels)
print("Format:", audio_info.subtype)


Audio file: meeting_test.wav
Duration: 60.0 seconds
Sample rate: 16000 Hz
Channels: 1
Format: PCM_16


In [9]:
from pyannote.audio.pipelines.utils.hook import ProgressHook

print("Starting speaker diarization...")

with ProgressHook() as hook:
    output = pipeline(
        audio_path,
        num_speakers=2,
        hook=hook
    )

print("Speaker diarization completed.")

Output()

/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: 
TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(

Starting speaker diarization...


/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of
freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output 
numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1913.)
  std = sequences.std(dim=-1, correction=1)

Speaker diarization completed.


In [10]:
print("SPEAKER DIARIZATION RESULTS")
print("=" * 60)

for turn, speaker in output.speaker_diarization:
    print(
        f"{speaker}: "
        f"{turn.start:.2f}s - {turn.end:.2f}s"
    )

SPEAKER DIARIZATION RESULTS
SPEAKER_00: 2.82s - 5.72s
SPEAKER_01: 8.57s - 10.49s
SPEAKER_01: 11.19s - 14.70s
SPEAKER_01: 15.29s - 16.52s
SPEAKER_00: 21.58s - 22.32s
SPEAKER_00: 23.37s - 26.96s
SPEAKER_00: 29.87s - 29.90s
SPEAKER_01: 29.90s - 31.32s
SPEAKER_01: 31.71s - 36.09s
SPEAKER_00: 38.07s - 41.88s
SPEAKER_00: 41.98s - 42.88s
SPEAKER_01: 44.41s - 47.97s
SPEAKER_00: 47.97s - 47.99s
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s
SPEAKER_00: 54.96s - 55.67s
SPEAKER_01: 56.82s - 58.91s


In [11]:
print("EXCLUSIVE SPEAKER DIARIZATION")
print("=" * 60)

for turn, speaker in output.exclusive_speaker_diarization:
    print(
        f"{speaker}: "
        f"{turn.start:.2f}s - {turn.end:.2f}s"
    )

EXCLUSIVE SPEAKER DIARIZATION
SPEAKER_00: 2.82s - 5.72s
SPEAKER_01: 8.57s - 10.49s
SPEAKER_01: 11.19s - 14.70s
SPEAKER_01: 15.29s - 16.52s
SPEAKER_00: 21.58s - 22.32s
SPEAKER_00: 23.37s - 26.96s
SPEAKER_00: 29.87s - 29.90s
SPEAKER_01: 29.90s - 31.32s
SPEAKER_01: 31.71s - 36.09s
SPEAKER_00: 38.07s - 41.88s
SPEAKER_00: 41.98s - 42.88s
SPEAKER_01: 44.41s - 47.97s
SPEAKER_00: 47.97s - 47.99s
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s
SPEAKER_00: 54.96s - 55.67s
SPEAKER_01: 56.82s - 58.91s


In [12]:
speakers = {
    str(speaker)
    for _, speaker in output.speaker_diarization
}

print("Number of speakers:", len(speakers))
print("Speaker IDs:", sorted(speakers))

Number of speakers: 2
Speaker IDs: ['SPEAKER_00', 'SPEAKER_01']


In [13]:
from pyannote.audio.pipelines.utils.hook import ProgressHook

print("Running automatic speaker detection...")

with ProgressHook() as hook:
    output_auto = pipeline(
        audio_path,
        hook=hook
    )

auto_speakers = {
    str(speaker)
    for _, speaker in output_auto.speaker_diarization
}

print("\nAutomatic detection result")
print("Number of speakers:", len(auto_speakers))
print("Speaker IDs:", sorted(auto_speakers))

Output()

Running automatic speaker detection...



Automatic detection result
Number of speakers: 2
Speaker IDs: ['SPEAKER_00', 'SPEAKER_01']


In [14]:
print("REGULAR SPEAKER DIARIZATION")

for turn, speaker in output_auto.speaker_diarization:
    print(
        f"{speaker}: "
        f"{turn.start:.2f}s - {turn.end:.2f}s"
    )

REGULAR SPEAKER DIARIZATION
SPEAKER_00: 2.82s - 5.72s
SPEAKER_01: 8.57s - 10.49s
SPEAKER_01: 11.19s - 14.70s
SPEAKER_01: 15.29s - 16.52s
SPEAKER_00: 21.58s - 22.32s
SPEAKER_00: 23.37s - 26.96s
SPEAKER_00: 29.87s - 29.90s
SPEAKER_01: 29.90s - 31.32s
SPEAKER_01: 31.71s - 36.09s
SPEAKER_00: 38.07s - 41.88s
SPEAKER_00: 41.98s - 42.88s
SPEAKER_01: 44.41s - 47.97s
SPEAKER_00: 47.97s - 47.99s
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s
SPEAKER_00: 54.96s - 55.67s
SPEAKER_01: 56.82s - 58.91s


In [15]:
from collections import Counter

turns = [
    {
        "speaker": str(speaker),
        "start": float(turn.start),
        "end": float(turn.end),
        "duration": float(turn.end - turn.start),
    }
    for turn, speaker in output_auto.speaker_diarization
]

speakers = sorted({item["speaker"] for item in turns})
short_segments = [
    item for item in turns
    if item["duration"] < 0.1
]

print("DIARIZATION QUALITY CHECK")
print("=" * 50)

print("Number of speakers:", len(speakers))
print("Speaker IDs:", speakers)
print("Total segments:", len(turns))

print("\nSegments per speaker:")
for speaker, count in Counter(
    item["speaker"] for item in turns
).items():
    print(f"{speaker}: {count}")

print("\nSuspiciously short segments (< 0.1s):")
for item in short_segments:
    print(item)

print("\nTotal short segments:", len(short_segments))

DIARIZATION QUALITY CHECK
Number of speakers: 2
Speaker IDs: ['SPEAKER_00', 'SPEAKER_01']
Total segments: 20

Segments per speaker:
SPEAKER_00: 11
SPEAKER_01: 9

Suspiciously short segments (< 0.1s):
{'speaker': 'SPEAKER_00', 'start': 29.86596875, 'end': 29.89971875, 'duration': 0.03375000000000128}
{'speaker': 'SPEAKER_00', 'start': 47.97284375, 'end': 47.98971875, 'duration': 0.016874999999998863}
{'speaker': 'SPEAKER_01', 'start': 49.930343750000006, 'end': 49.947218750000005, 'duration': 0.016874999999998863}
{'speaker': 'SPEAKER_00', 'start': 50.470343750000005, 'end': 50.487218750000004, 'duration': 0.016874999999998863}
{'speaker': 'SPEAKER_01', 'start': 50.487218750000004, 'end': 50.52096875, 'duration': 0.033749999999997726}

Total short segments: 5


In [16]:
import soundfile as sf
from IPython.display import Audio, display

# Đọc file audio gốc
waveform, sample_rate = sf.read(audio_path)

# Chuyển stereo thành mono nếu cần
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1)

# Lấy các phân đoạn ngắn từ kết quả diarization tự động
short_segments = [
    {
        "speaker": str(speaker),
        "start": float(turn.start),
        "end": float(turn.end),
        "duration": float(turn.end - turn.start)
    }
    for turn, speaker in output_auto.speaker_diarization
    if turn.end - turn.start < 0.1
]

# Phát lại audio xung quanh từng phân đoạn
audio_duration = len(waveform) / sample_rate

for index, segment in enumerate(short_segments, start=1):
    start = max(0, segment["start"] - 0.8)
    end = min(audio_duration, segment["end"] + 0.8)

    start_index = int(start * sample_rate)
    end_index = int(end * sample_rate)

    audio_clip = waveform[start_index:end_index]

    print(
        f"Segment {index}: {segment['speaker']} | "
        f"{segment['start']:.3f}s - {segment['end']:.3f}s"
    )

    display(Audio(audio_clip, rate=sample_rate))

Segment 1: SPEAKER_00 | 29.866s - 29.900s


Segment 2: SPEAKER_00 | 47.973s - 47.990s


Segment 3: SPEAKER_01 | 49.930s - 49.947s


Segment 4: SPEAKER_00 | 50.470s - 50.487s


Segment 5: SPEAKER_01 | 50.487s - 50.521s


In [17]:
# Kiểm tra các phân đoạn xung quanh 5 đoạn bất thường

for short in short_segments:
    print("\n" + "=" * 60)
    print(
        f"Đoạn cần kiểm tra: {short['speaker']} "
        f"({short['start']:.2f}s - {short['end']:.2f}s)"
    )

    context_start = short["start"] - 2
    context_end = short["end"] + 2

    for turn, speaker in output_auto.speaker_diarization:
        if turn.end >= context_start and turn.start <= context_end:
            print(
                f"{speaker}: "
                f"{turn.start:.2f}s - {turn.end:.2f}s"
            )


Đoạn cần kiểm tra: SPEAKER_00 (29.87s - 29.90s)
SPEAKER_00: 29.87s - 29.90s
SPEAKER_01: 29.90s - 31.32s
SPEAKER_01: 31.71s - 36.09s

Đoạn cần kiểm tra: SPEAKER_00 (47.97s - 47.99s)
SPEAKER_01: 44.41s - 47.97s
SPEAKER_00: 47.97s - 47.99s
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s

Đoạn cần kiểm tra: SPEAKER_01 (49.93s - 49.95s)
SPEAKER_01: 44.41s - 47.97s
SPEAKER_00: 47.97s - 47.99s
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s

Đoạn cần kiểm tra: SPEAKER_00 (50.47s - 50.49s)
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s

Đoạn cần kiểm tra: SPEAKER_01 (50.49s - 50.52s)
SPEAKER_01: 49.93s - 49.95s
SPEAKER_00: 49.95s - 50.30s
SPEAKER_00: 50.47s - 50.49s
SPEAKER_01: 50.49s - 50.52s
SPEAKER_00: 50.52s - 54.77s


In [18]:
import json
import soundfile as sf
from google.colab import files

def convert_annotation(annotation):
    segments = []

    for turn, speaker in annotation:
        duration = turn.end - turn.start

        segments.append({
            "start": round(float(turn.start), 3),
            "end": round(float(turn.end), 3),
            "speaker_id": str(speaker),
            "duration": round(float(duration), 3),
            "short_segment_flag": duration < 0.1
        })

    return segments


audio_info = sf.info(audio_path)

regular_segments = convert_annotation(
    output_auto.speaker_diarization
)

exclusive_segments = convert_annotation(
    output_auto.exclusive_speaker_diarization
)

speaker_ids = sorted({
    segment["speaker_id"]
    for segment in regular_segments
})

result = {
    "model": "pyannote/speaker-diarization-community-1",
    "audio_file": audio_path,
    "audio_duration": audio_info.duration,
    "sample_rate": audio_info.samplerate,
    "num_speakers": len(speaker_ids),
    "speaker_ids": speaker_ids,
    "configuration": {
        "mode": "automatic",
        "num_speakers": None
    },
    "regular_diarization": regular_segments,
    "exclusive_diarization": exclusive_segments
}

with open("diarization_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Export completed.")
print("Speakers:", speaker_ids)
print("Regular segments:", len(regular_segments))
print("Exclusive segments:", len(exclusive_segments))
print("Flagged segments:", sum(
    segment["short_segment_flag"]
    for segment in regular_segments
))

files.download("diarization_result.json")

Export completed.
Speakers: ['SPEAKER_00', 'SPEAKER_01']
Regular segments: 20
Exclusive segments: 20
Flagged segments: 5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## KẾT QUẢ THỰC NGHIỆM SPEAKER DIARIZATION

### 1. Môi trường thực nghiệm

- Nền tảng: Google Colab.
- GPU: NVIDIA Tesla T4.
- PyTorch: 2.14.1+cu130.
- pyannote.audio: 4.0.7.
- Model: pyannote/speaker-diarization-community-1.

### 2. Dữ liệu thử nghiệm

- Dữ liệu: File ghi âm cuộc hội thoại tiếng Việt có hai người tham gia.
- Thời lượng: Khoảng 60 giây.
- Số lượng người nói thực tế: 2.

### 3. Kết quả nhận dạng

Khi cung cấp trước số lượng người nói bằng tham số num_speakers=2, mô hình tạo được hai Speaker ID.

Trong lần thử nghiệm tự động, không cung cấp trước số lượng người nói, mô hình cũng phát hiện được hai nhóm giọng nói:

- SPEAKER_00: 11 phân đoạn.
- SPEAKER_01: 9 phân đoạn.
- Tổng số phân đoạn: 20.

### 4. Kiểm tra chất lượng phân đoạn

Phát hiện 5 phân đoạn có thời lượng nhỏ hơn 0,1 giây.

Các phân đoạn này được đánh dấu để kiểm tra thủ công. Khi nghe lại những vùng âm thanh mở rộng, ghi nhận có hai giọng nói luân phiên.

Không loại bỏ hoặc tự ý thay đổi kết quả gốc của mô hình.

### 5. Kết luận

Mô hình pyannote/speaker-diarization-community-1 đã thực hiện được việc phân nhóm người nói trên dữ liệu hội thoại tiếng Việt thử nghiệm.

Kết quả cung cấp Speaker ID và các mốc thời gian bắt đầu, kết thúc của từng lượt nói.

Đây là cơ sở thực nghiệm để tiếp tục nghiên cứu việc kết hợp kết quả Speaker Diarization với transcript được tạo bởi PhoWhisper ASR.

Lưu ý: Kết quả thử nghiệm chưa đủ để khẳng định độ chính xác tổng quát của mô hình trên các cuộc họp tiếng Việt khác nhau.